In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("ETL_pipeline").getOrCreate()
print("spark session ready")

spark session ready


In [ ]:
from pyspark.sql.functions import (
    col, sum, avg, max, month, year, to_date, regexp_replace, when, upper
)
import dlt
@dlt.table(
    name= "bronze_expenses"
)
def bronze_expenses():
  expenses=[
  (1,1,1,"UPI",250,"2026-05-01"),
  (2,1,2,"CASH",12000,"2026-05-02"),
  (3,1,4,"CARD",1500,"2026-05-05"),
  (4,2,1,"CARD",300,"2026-05-01"),
  (5,2,6,"UPI",2000,"2026-05-03"),
  (6,1,3,"UPI",400,"2026-06-01"),
  (7,1,5,"CASH",700,"2026-06-06"),
  (8,2,3,"CARD",999,"2026-06-05"),
  (9,3,1,"UPI",450,"2026-05-10"),
  (10,3,2,"BANK_TRANSFER",13000,"2026-05-02"),
  (11,3,4,"CARD",2200,"2026-05-18"),
  (12,4,6,"UPI",1800,"2026-05-11"),
  (13,4,1,"CASH",350,"2026-05-13"),
  (14,4,3,"CARD",500,"2026-06-02"),
  (15,2,2,"BANK_TRANSFER",12500,"2026-06-02"),
  (16,2,5,"CASH",900,"2026-06-08"),
  (17,5,1,"UPI",600,"2026-05-06"),
  (18,5,4,"CARD",2500,"2026-05-15"),
  (19,5,6,"CASH",3000,"2026-05-21"),
  (20,5,3,"UPI",450,"2026-06-03"),
  (21,1,6,"CARD",850,"2026-06-07"),
  (22,1,4,"CARD",3000,"2026-06-15"),
  (23,3,5,"CASH",750,"2026-06-10"),
  (24,3,6,"UPI",2100,"2026-06-12"),
  (25,4,2,"BANK_TRANSFER",14000,"2026-06-01"),
  (26,4,4,"CARD",5000,"2026-06-14"),
  (27,2,1,"UPI",200,"2026-07-01"),
  (28,2,3,"CARD",450,"2026-07-02"),
  (29,3,6,"UPI",60000,"2026-07-05"),
  (30,1,2,"BANK_TRANSFER",45000,"2026-07-06")]

  columns= ["expense_id","user_id","category_id","payment_mode","amount","expense_date"]

  expense_df = spark.createDataFrame(expenses, columns)

  return expense_df

@dlt.table(
    name="bronze_users"
)
def  bronze_users():
  users=[
  (1,"Rahul","Sharma","rahul@gmail.com","rahul123",60000)
  (2,"Anjana","Arora","anjana@gmail.com","anjana456",55000)
  (3,"John","Doe","john@gmail.com","john123",75000)
  (4,"Nikitha","Ramesh","nikitha@gmail.com","nikitha456",48000)
  (5,"Sebastian","Paul","sebastian@gmail.com","sebastian123",82000)]

  column=["user_id","f_name","l_name","email","password","monthly_income"]
  users_df = spark.createDataFrame(users, column)

  return users_df


In [ ]:
@dlt.table(
    name= "silver_expenses"
)

def silver_expenses():
  expense_df=dlt.read("bronze_expenses")
  expense_df = expense_df.withColumn("payment_mode", upper(col("payment_mode")))
  expense_df = expense_df.withColumn("expense_date", to_date(col("expense_date"), "yyyy-MM-dd"))
  expense_df = expense_df.filter(col("expense_date").isNotNull())

  return expense_df

@dlt.table(
    name="silver_users"
)
def silver_users():
    users_df = dlt.read("bronze_users")

    users_df = users_df.withColumn("f_name", initcap(col("f_name"))) \
           .withColumn("l_name", initcap(col("l_name")))

    return users_df


In [ ]:
@dlt.table(
    name="gold_monthly_summary"
)
def gold_monthly_summary():
    expenses_df = dlt.read("silver_expenses")
    users_df = dlt.read("silver_users")

    joined_df = expenses_df.join(users_df, on="user_id", how="inner")

    monthly_summary = joined_df.groupBy(
        "user_id",
        year("expense_date").alias("year"),
        month("expense_date").alias("month")
    ).agg(
        sum("amount").alias("monthly_spend")
    )

    monthly_summary = monthly_summary.withColumn(
        "monthly_savings",
        col("monthly_income") - col("monthly_spend")
    )

    monthly_summary = monthly_summary.withColumn(
        "alert",
        when(col("monthly_spend") > col("monthly_income"), "OVESPENDING")
        .when(col("monthly_savings") < 5000, "LOW_SAVINGS")
        .otherwise("NORMAL")
    )

    return monthly_summary